In [0]:

import importlib
import configs.constant as constants
import utils.transformation_function as transformation_functions
importlib.reload(constants)
importlib.reload(transformation_functions)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from configs.constant import * 


df_bronze = spark.readStream.format("delta") \
    .load(f"abfss://{BRONZE_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/kafka_data/")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# 🔹 UPDATED SCHEMA (IMPORTANT 🔥)
schema = StructType([
    StructField("event_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("session_id", StringType()),
    StructField("event_type", StringType()),
    StructField("product_id", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
    StructField("region", StringType()),
    StructField("device_type", StringType()),
    StructField("app_version", StringType()),
    
    # 🔥 NEW FIELDS (must add)
    StructField("price", DoubleType()),
    StructField("quantity", IntegerType()),
    StructField("cart_total", DoubleType()),
    StructField("amount", DoubleType()),
    StructField("payment_method", StringType())
])

# 🔹 PARSE JSON
df_parsed = df_bronze \
    .withColumn("decoded", col("value").cast("string")) \
    .withColumn("json", from_json(col("decoded"), schema)) \
    .select(
        "json.*",
        "decoded",
        "ingestion_timestamp"
    )

# 🔹 TIMESTAMP FIX (VERY IMPORTANT ⚠️)
df_parsed = df_parsed \
    .withColumn("event_timestamp", to_timestamp("timestamp")) \
    .withColumn("event_date", to_date("event_timestamp")) \
    .withColumn("event_hour", hour("event_timestamp")) \
    .withColumn("event_week", weekofyear("event_timestamp"))



In [0]:
spark.conf.set(
  f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
  STORAGE_ACCOUNT_ACCESS_KEY
)

In [0]:
display(df_parsed)

In [0]:

df_parsed = transformation_functions.Add_newField(df_parsed)

In [0]:

df_transformed  = transformation_functions.Parse_Time(df_parsed)

In [0]:

df_clean = transformation_functions.Filtering(df_transformed)

In [0]:

df_final = transformation_functions.Remove_Duplicate(df_clean)

In [0]:
df_final.isStreaming  # should be True

In [0]:
df_final.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://{SILVER_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/") \
    .partitionBy("event_date", "event_type") \
    .start(f"abfss://{SILVER_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/clean_data/")